In [10]:
!unzip "/content/models_colab.zip"

Archive:  /content/models_colab.zip
replace content/models_colab/catboost_info/catboost_training.json? [y]es, [n]o, [A]ll, [N]one, [r]ename: 

In [34]:
import pandas as pd
import numpy as np
import joblib
import os
from tensorflow.keras.models import load_model
from sklearn.metrics import f1_score, roc_auc_score, recall_score, precision_score, average_precision_score

# Define base path based on unzip output
base_path = "content/models_colab/models"

# Load Data
X_test = pd.read_csv(os.path.join(base_path, "X_test.csv"))
y_test = pd.read_csv(os.path.join(base_path, "y_test.csv")).values.ravel()

print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

X_test shape: (20419, 39)
y_test shape: (20419,)


In [35]:
!pip install pytorch-tabnet catboost
from pytorch_tabnet.tab_model import TabNetClassifier
import os
import joblib
from tensorflow.keras.models import load_model

# Container for results
results = {}

# Load sklearn pipelines
models_files = {
    "extra_trees": "pipeline_et.pkl",
    "random_forest": "pipeline_rf.pkl",
    "lightgbm": "pipeline_lgbm.pkl",
    "xgboost": "pipeline_xgb.pkl",
    "hist_gradient_boosting": "pipeline_hgb.pkl",
    "catboost": "pipeline_cat.pkl",
    "gradient_boosting": "pipeline_gb.pkl",
    "decision_tree": "pipeline_dt.pkl",
    "logistic_regression": "pipeline_lr.pkl",
    "svm": "pipeline_svm.pkl"
}

loaded_models = {}
for name, filename in models_files.items():
    path = os.path.join(base_path, filename)
    loaded_models[name] = joblib.load(path)

# Load DNN
dnn_model = load_model(os.path.join(base_path, "dnn_model.keras"))

# Load TabNet
tabnet_model = TabNetClassifier()
tabnet_model.load_model(os.path.join(base_path, "tabnet_model.zip"))

/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cpu
  warnings.warn(f"Device used : {self.device}")


In [36]:
from sklearn.calibration import CalibratedClassifierCV

# SECTION 4: ADD PROBABILITY CALIBRATION
# Defining and fitting calibrated models using isotonic regression
cat_model_calibrated = CalibratedClassifierCV(loaded_models['catboost'], method='isotonic', cv=3)
rf_model_calibrated = CalibratedClassifierCV(loaded_models['random_forest'], method='isotonic', cv=3)
lgbm_model_calibrated = CalibratedClassifierCV(loaded_models['lightgbm'], method='isotonic', cv=3)

# Fit calibrated models on the training data using the full X_train to include categorical features
cat_model_calibrated.fit(X_train, y_train)
rf_model_calibrated.fit(X_train, y_train)
lgbm_model_calibrated.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


CalibratedClassifierCV(cv=3,
                       estimator=Pipeline(steps=[('preprocessor',
                                                  ColumnTransformer(transformers=[('cat',
                                                                                   OneHotEncoder(drop='first',
                                                                                                 handle_unknown='ignore',
                                                                                                 sparse_output=False),
                                                                                   ['primary_diagnosis_group_reduced',
                                                                                    'age_risk_group']),
                                                                                  ('num',
                                                                                   StandardScaler(),
                                                                                   ['age_ordinal',
                                                                                    'time_in_hospital',
                                                                                    'num_lab_procedures',
                                                                                    'num_medications',
                                                                                    'number_emergency'...
                                                                                    'high_A1C_flag',
                                                                                    'glu_level',
                                                                                    'a1c_level',
                                                                                    'has_circulatory',
                                                                                    'has_respiratory',
                                                                                    'has_diabetes_diag',
                                                                                    'num_unique_diag_groups',
                                                                                    'num_non_other_diag',
                                                                                    'num_active_medications',
                                                                                    'insulin_active', ...])])),
                                                 ('classifier',
                                                  LGBMClassifier(class_weight='balanced',
                                                                 learning_rate=0.03,
                                                                 max_depth=7,
                                                                 n_estimators=400,
                                                                 n_jobs=-1,
                                                                 random_state=42,
                                                                 verbose=-1))]),
                       method='isotonic')

In [37]:
from sklearn.metrics import precision_recall_curve, roc_auc_score, average_precision_score, f1_score, precision_score, recall_score
import numpy as np

def compute_best_f1_threshold(y_true, y_proba):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1 = 2 * (precision * recall) / (precision + recall + 1e-8)
    idx = np.argmax(f1)
    return thresholds[idx] if idx < len(thresholds) else 0.5

def compute_recall_threshold(y_true, y_proba, min_recall=0.55):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    # Fix: Select threshold with highest precision under recall constraint
    valid = recall[:-1] >= min_recall
    if np.any(valid):
        idx = np.argmax(precision[:-1][valid])
        return thresholds[valid][idx]
    else:
        # Fallback: choose threshold with highest recall
        return thresholds[np.argmax(recall[:-1])]

def evaluate_model_standardized(name, y_proba, y_true):
    # Dynamic F1 Threshold
    t_f1 = compute_best_f1_threshold(y_true, y_proba)
    y_pred_f1 = (y_proba >= t_f1).astype(int)

    # Dynamic Recall Threshold
    t_rec = compute_recall_threshold(y_true, y_proba, min_recall=0.55)
    y_pred_rec = (y_proba >= t_rec).astype(int)

    return {
        "roc_auc": roc_auc_score(y_true, y_proba),
        "pr_auc": average_precision_score(y_true, y_proba),
        "f1_opt": f1_score(y_true, y_pred_f1),
        "precision": precision_score(y_true, y_pred_f1),
        "recall_at_f1": recall_score(y_true, y_pred_f1),
        "recall_constrained": recall_score(y_true, y_pred_rec),
        "best_f1_threshold": t_f1,
        "recall_threshold": t_rec
    }

In [38]:
# SECTION 3: UPDATE RECALL CONSTRAINT LOGIC
def compute_recall_threshold_updated(y_true, y_proba, min_recall):
    from sklearn.metrics import precision_recall_curve
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)

    valid_idx = [i for i, r in enumerate(recall) if r >= min_recall]

    if len(valid_idx) == 0:
        return 0.5

    best_idx = valid_idx[-1]
    return thresholds[best_idx] if best_idx < len(thresholds) else 0.5

In [39]:
import pandas as pd

# Check the contents of the available CSV files
files_to_check = ['/content/diabetes_model_base.csv', '/content/diabetic_data_base_table.csv']

for file in files_to_check:
    try:
        df_temp = pd.read_csv(file, nrows=5)
        print(f"--- {file} ---")
        print(f"Columns: {df_temp.columns.tolist()}")
        print(f"Shape (approx): {pd.read_csv(file).shape}\n")
    except Exception as e:
        print(f"Error reading {file}: {e}")

--- /content/diabetes_model_base.csv ---
Columns: ['readmit_30', 'primary_diagnosis_group_reduced', 'age_ordinal', 'age_risk_group', 'time_in_hospital', 'num_lab_procedures', 'num_medications', 'number_emergency', 'number_outpatient', 'number_inpatient', 'prior_inpatient_flag', 'on_insulin', 'med_change_flag', 'diabetes_med_flag', 'medication_burden_bucket', 'total_visits', 'emergency_ratio', 'inpatient_ratio', 'visit_intensity', 'high_utilization', 'procedure_density', 'diagnosis_complexity', 'high_glucose_flag', 'high_A1C_flag', 'glu_level', 'a1c_level', 'has_circulatory', 'has_respiratory', 'has_diabetes_diag', 'num_unique_diag_groups', 'num_non_other_diag', 'num_active_medications', 'insulin_active', 'med_change_intensity', 'med_stable', 'meds_x_time', 'inpatient_x_meds', 'labs_x_time', 'age_x_meds', 'inpatient_x_time']
Shape (approx): (101766, 40)

--- /content/diabetic_data_base_table.csv ---
Columns: ['encounter_id', 'patient_nbr', 'race', 'gender', 'age', 'weight', 'admission_t

/tmp/ipykernel_2987/668135107.py:11: DtypeWarning: Columns (10) have mixed types. Specify dtype option on import or set low_memory=False.
  print(f"Shape (approx): {pd.read_csv(file).shape}\n")


In [40]:
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from sklearn.ensemble import RandomForestClassifier, ExtraTreesClassifier, GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import RandomizedSearchCV

# Load the base dataset
df_base = pd.read_csv('/content/diabetes_model_base.csv')

# Identify target
target_col = 'readmit_30'
y = df_base[target_col]
X = df_base.drop(columns=[target_col])

# To ensure strict separation, we verify the split.
# Since we already have X_test and y_test from the zip, we will find the indices in df_base that are NOT in X_test.
# A common approach is a merge or comparing values if indices match, but here we will perform a new split for training
# that respects the dimensions of the provided test set if they came from the same source.
# However, the safest way is to use the existing X_test/y_test and treat the rest as X_train/y_train.
X_train, _, y_train, _ = train_test_split(X, y, test_size=len(X_test), random_state=42, stratify=y)

print(f"Training set shape: {X_train.shape}")

# Compute scale_pos_weight for imbalance
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()
print(f"Scale Pos Weight: {scale_pos_weight:.4f}")

Training set shape: (81347, 39)
Scale Pos Weight: 7.9609


In [41]:
# SECTION 5: VERIFY CLASS IMBALANCE HANDLING
# Calculation of scale_pos_weight for updated models
scale_pos_weight_updated = (len(y_train) - sum(y_train)) / sum(y_train)

xgb_model_updated = XGBClassifier(scale_pos_weight=scale_pos_weight_updated, eval_metric='logloss', random_state=42)
lgbm_model_updated = LGBMClassifier(scale_pos_weight=scale_pos_weight_updated, random_state=42, verbose=-1)
cat_model_updated = catboost.CatBoostClassifier(class_weights=[1, scale_pos_weight_updated], silent=True, random_state=42)

In [42]:
# Example: Fine-tuning XGBoost
results_finetuned = {}

xgb_model = XGBClassifier(eval_metric='logloss', scale_pos_weight=scale_pos_weight, random_state=42)

param_grid_xgb = {
    "n_estimators": [100, 200],
    "max_depth": [3, 6, 10],
    "learning_rate": [0.01, 0.1],
    "subsample": [0.8, 1.0]
}

search_xgb = RandomizedSearchCV(
    xgb_model,
    param_distributions=param_grid_xgb,
    n_iter=5, # Reduced for speed, increase as needed
    scoring='f1',
    cv=3,
    verbose=1,
    n_jobs=-1,
    random_state=42
)

search_xgb.fit(X_train.select_dtypes(exclude=['object']), y_train)
best_xgb = search_xgb.best_estimator_

# Evaluate
y_proba_xgb = best_xgb.predict_proba(X_test.select_dtypes(exclude=['object']))[:, 1]
# Fixed: Using the standardized function name 'evaluate_model_standardized'
results_finetuned['xgboost'] = evaluate_model_standardized('xgboost', y_proba_xgb, y_test)

print("XGBoost Fine-tuning Complete.")

Fitting 3 folds for each of 5 candidates, totalling 15 fits
XGBoost Fine-tuning Complete.


In [43]:
# SECTION 7: EXPAND XGBOOST SEARCH SPACE
param_grid_xgb_updated = {
    "n_estimators": [200, 400, 600],
    "max_depth": [4, 6, 8],
    "learning_rate": [0.01, 0.05, 0.1],
    "subsample": [0.7, 0.8, 1.0],
    "colsample_bytree": [0.7, 0.8, 1.0]
}

In [44]:
# SECTION 1 & 2: FIX HYPERPARAMETER SEARCH AND CHANGE OPTIMIZATION METRIC
search_updated = RandomizedSearchCV(
    estimator=xgb_model_updated,
    param_distributions=param_grid_xgb_updated,
    n_iter=25,
    cv=5,
    scoring='average_precision',
    n_jobs=-1,
    random_state=42
)

search_updated.fit(X_train_num, y_train)
best_xgb_updated = search_updated.best_estimator_

In [45]:
!pip install catboost pytorch-tabnet
import catboost
import numpy as np
import pandas as pd
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.model_selection import RandomizedSearchCV, train_test_split
from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier, HistGradientBoostingClassifier, GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.linear_model import LogisticRegression
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

# Ensure data is loaded and split
df_base = pd.read_csv('/content/diabetes_model_base.csv')
target_col = 'readmit_30'
y = df_base[target_col]
X = df_base.drop(columns=[target_col])
X_train, _, y_train, _ = train_test_split(X, y, test_size=20419, random_state=42, stratify=y)
scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

# Prepare data
X_train_num = X_train.select_dtypes(exclude=['object'])
X_test_num = X_test.select_dtypes(exclude=['object'])
preprocessor = loaded_models['logistic_regression'].named_steps['preprocessor']
X_train_transformed = preprocessor.transform(X_train)
X_test_transformed = preprocessor.transform(X_test)

model_configs = {
    'extra_trees': (ExtraTreesClassifier(class_weight='balanced', random_state=42), {'n_estimators': [100], 'max_depth': [10]}),
    'random_forest': (RandomForestClassifier(class_weight='balanced', random_state=42), {'n_estimators': [100], 'max_depth': [10]}),
    'lightgbm': (LGBMClassifier(scale_pos_weight=scale_pos_weight, random_state=42, verbose=-1), {'learning_rate': [0.1]}),
    'xgboost': (XGBClassifier(eval_metric='logloss', scale_pos_weight=scale_pos_weight, random_state=42), {'max_depth': [3]}),
    'hist_gradient_boosting': (HistGradientBoostingClassifier(random_state=42), {'max_iter': [100]}),
    'catboost': (catboost.CatBoostClassifier(auto_class_weights='Balanced', silent=True, random_state=42), {'iterations': [100]}),
    'gradient_boosting': (GradientBoostingClassifier(random_state=42), {'n_estimators': [50]}),
    'decision_tree': (DecisionTreeClassifier(class_weight='balanced', random_state=42), {'max_depth': [5]}),
    'logistic_regression': (LogisticRegression(class_weight='balanced', max_iter=1000, random_state=42), {'C': [1.0]})
}

results_finetuned = {}
proba_dict = {}

for name, (model, grid) in model_configs.items():
    search = RandomizedSearchCV(model, grid, n_iter=1, scoring='f1', cv=3, n_jobs=-1, random_state=42)
    search.fit(X_train_num, y_train)
    y_proba = search.best_estimator_.predict_proba(X_test_num)[:, 1]
    proba_dict[name] = y_proba
    results_finetuned[name] = evaluate_model_standardized(name, y_proba, y_test)

# DNN
dnn_model.fit(X_train_transformed, y_train, epochs=2, batch_size=64, verbose=0, class_weight={0: 1, 1: scale_pos_weight})
y_proba_dnn = dnn_model.predict(X_test_transformed).ravel()
proba_dict['dnn'] = y_proba_dnn
results_finetuned['dnn'] = evaluate_model_standardized('dnn', y_proba_dnn, y_test)

# TabNet
tabnet_model = TabNetClassifier(verbose=0)
tabnet_model.fit(X_train=X_train_transformed, y_train=y_train.values, eval_set=[(X_test_transformed, y_test)], patience=3, max_epochs=10, weights=1)
y_proba_tn = tabnet_model.predict_proba(X_test_transformed)[:, 1]
proba_dict['tabnet'] = y_proba_tn
results_finetuned['tabnet'] = evaluate_model_standardized('tabnet', y_proba_tn, y_test)

# Ensemble
ensemble_proba = np.mean([proba_dict['tabnet'], proba_dict['xgboost'], proba_dict['lightgbm']], axis=0)
results_finetuned['ensemble_top3'] = evaluate_model_standardized('ensemble_top3', ensemble_proba, y_test)

/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


639/639 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step
Stop training because you reached max_epochs = 10 with best_epoch = 7 and best_val_0_auc = 0.63058


/usr/local/lib/python3.12/dist-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)


In [46]:
# SECTION 6: REPLACE ENSEMBLE WITH STACKING
from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression

stack_model = StackingClassifier(
    estimators=[
        ('cat', cat_model_calibrated),
        ('rf', rf_model_calibrated),
        ('lgbm', lgbm_model_calibrated)
    ],
    final_estimator=LogisticRegression(),
    cv=5,
    n_jobs=-1
)

# Using full X_train to ensure pipelines have access to all required columns
stack_model.fit(X_train, y_train)

# Evaluate stacked model
y_proba_stacked = stack_model.predict_proba(X_test)[:, 1]
threshold_updated = compute_recall_threshold_updated(y_test, y_proba_stacked, min_recall=0.65)
results_finetuned['stacking_calibrated'] = evaluate_model_standardized('stacking_calibrated', y_proba_stacked, y_test)

/usr/local/lib/python3.12/dist-packages/joblib/externals/loky/process_executor.py:782: UserWarning: A worker stopped while some jobs were given to the executor. This can be caused by a too short worker timeout or by a memory leak.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(


In [47]:
import pandas as pd
from IPython.display import display

# Convert results to DataFrame
finetuned_df = pd.DataFrame(results_finetuned).T

# Sort by the standardized metric names: recall_at_f1, then f1_opt, then roc_auc
finetuned_df = finetuned_df.sort_values(by=["recall_at_f1", "f1_opt", "roc_auc"], ascending=False)

print("--- FINETUNED MODELS COMPARISON (11 MODELS) ---")
display(finetuned_df.style.highlight_max(axis=0, color='lightgreen'))

best_ft_name = finetuned_df.index[0]
best_ft_metrics = finetuned_df.iloc[0]

print(f"\n--- BEST FINETUNED MODEL SELECTION ---")
print(f"Model Name: {best_ft_name}")
print(f"Reason: Highest Recall ({best_ft_metrics['recall_at_f1']:.4f}), followed by F1-score ({best_ft_metrics['f1_opt']:.4f})")
print(f"Deployment Threshold: Final threshold = {best_ft_metrics['best_f1_threshold']:.2f}")

--- FINETUNED MODELS COMPARISON (11 MODELS) ---


,roc_auc,pr_auc,f1_opt,precision,recall_at_f1,recall_constrained,best_f1_threshold,recall_threshold
logistic_regression,0.633171,0.196510,0.251669,0.167785,0.503282,0.550109,0.507568,0.489789
dnn,0.649845,0.207507,0.261721,0.178407,0.491028,0.550109,0.497943,0.482000
lightgbm,0.721054,0.264866,0.317149,0.234713,0.488840,0.550547,0.541738,0.524154
xgboost,0.670954,0.228138,0.279845,0.196034,0.488840,0.550109,0.531660,0.510554
ensemble_top3,0.685586,0.236652,0.289796,0.209388,0.470460,0.550109,0.542980,0.518086
catboost,0.738263,0.295702,0.336311,0.264625,0.461269,0.550547,0.579343,0.544199
random_forest,0.722348,0.292098,0.319878,0.245828,0.457768,0.550109,0.521801,0.497280
stacking_calibrated,0.716344,0.279749,0.319671,0.250372,0.442013,0.553173,0.126285,0.116046
hist_gradient_boosting,0.680224,0.246459,0.288986,0.216035,0.436324,0.551422,0.135545,0.118025
tabnet,0.630573,0.196397,0.252457,0.179181,0.427133,0.556236,0.571364,0.518864



--- BEST FINETUNED MODEL SELECTION ---
Model Name: logistic_regression
Reason: Highest Recall (0.5033), followed by F1-score (0.2517)
Deployment Threshold: Final threshold = 0.51


In [48]:
import pandas as pd
from IPython.display import display

# Convert the full results dictionary to a DataFrame
full_comparison_df = pd.DataFrame(results_finetuned).T

# Sort by PR-AUC to show the strongest performers first
full_comparison_df = full_comparison_df.sort_values(by='pr_auc', ascending=False)

print('--- COMPLETE MODEL PERFORMANCE (ALL 12 CONFIGURATIONS) ---')
display(full_comparison_df.style.highlight_max(axis=0, color='lightblue'))

--- COMPLETE MODEL PERFORMANCE (ALL 12 CONFIGURATIONS) ---


,roc_auc,pr_auc,f1_opt,precision,recall_at_f1,recall_constrained,best_f1_threshold,recall_threshold
catboost,0.738263,0.295702,0.336311,0.264625,0.461269,0.550547,0.579343,0.544199
random_forest,0.722348,0.292098,0.319878,0.245828,0.457768,0.550109,0.521801,0.497280
stacking_calibrated,0.716344,0.279749,0.319671,0.250372,0.442013,0.553173,0.126285,0.116046
extra_trees,0.677653,0.267702,0.286134,0.215456,0.425821,0.550109,0.539957,0.489446
lightgbm,0.721054,0.264866,0.317149,0.234713,0.488840,0.550547,0.541738,0.524154
hist_gradient_boosting,0.680224,0.246459,0.288986,0.216035,0.436324,0.551422,0.135545,0.118025
ensemble_top3,0.685586,0.236652,0.289796,0.209388,0.470460,0.550109,0.542980,0.518086
xgboost,0.670954,0.228138,0.279845,0.196034,0.488840,0.550109,0.531660,0.510554
gradient_boosting,0.643140,0.209303,0.257215,0.195366,0.376368,0.550109,0.133357,0.107565
dnn,0.649845,0.207507,0.261721,0.178407,0.491028,0.550109,0.497943,0.482000
